<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_1/notebooks/W2L2_Colocalization_analysis__STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W2 L2 — Quantitative Cell Density and Colocalization Analysis

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

In this notebook you'll process a real confocal microscopy image (a `.czi` file from a Zeiss microscope) to find and count **proliferating oligodendrocyte-lineage cells**.

The image has three fluorescent channels:

| Channel | Stain | What it marks |
|---|---|---|
| DAPI | nuclear counterstain | the nucleus of *every* cell |
| **AF488** (green) | **Olig2** | **oligodendrocyte-lineage** cells (OPCs and oligodendrocytes) |
| **AF647** (far-red) | **Ki67** | cells that are **actively dividing** |

Olig2 tells you a cell belongs to the oligodendrocyte lineage. Ki67 tells you it's proliferating. A cell positive for **both** — an **Olig2⁺Ki67⁺** cell — is therefore a *proliferating oligodendrocyte-lineage cell*, usually a dividing OPC (mature oligodendrocytes rarely divide). Finding cells where the two signals overlap in the same nucleus is called **colocalization**, and that's what this notebook measures.

## What you'll be able to do by the end

1. Explain what Olig2 and Ki67 label, and why an Olig2⁺Ki67⁺ cell is a proliferating oligodendrocyte
2. Run a **segmentation pipeline**: thresholding, size filtering, and labelling
3. Explain how **mask overlap** identifies double-positive cells, and where that can mislead
4. Compute densities in **cells per mm²** and the **proliferation fraction**, and say why normalising matters
5. Judge how much to trust a result given the settings you chose

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

You'll see three kinds of things along the way:

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **⚙️ Try it** | Change the highlighted value, re-run the cell, and see what happens. |
| **💬 Discussion** | Double-click the cell and type your answer. We'll discuss these in class. |

If something breaks, first check that you ran **every cell above it**.

---

### Submitting

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

1. Answer every **💬 Discussion** cell (double-click, type, then Shift + Enter).
2. Click **Runtime → Run all** so every graph and answer shows up.
3. **File → Print → Save as PDF**, then upload the PDF to Canvas.

## Step 1 — Set up and fetch the image

This cell does three things: installs `czifile` (the library that reads Zeiss `.czi` files), imports the tools we need, and downloads the microscope image.

The image is about 85 MB, so the download takes a few seconds. It only happens once — if you re-run the cell, it will skip straight past.

> **If the download fails:** you can upload the file by hand instead. Click the 📁 folder icon in the left sidebar, then the upload button, and choose `1_slide_1_R.czi`. Then re-run this cell.

In [ ]:
!pip install czifile imagecodecs --quiet

import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from czifile import CziFile
from skimage.morphology import dilation, disk
from skimage.measure import label, regionprops

IMAGE_FILE = "1_slide_1_R.czi"
IMAGE_URL = "https://github.com/DrGPCR/NEUR201/releases/download/unit1-image/1_slide_1_R.czi"

if os.path.exists(IMAGE_FILE):
    print("Image already here — nothing to download.")
else:
    print("Downloading the image (about 85 MB)...")
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_FILE)
    print("Done.")

print("File size:", round(os.path.getsize(IMAGE_FILE) / 1e6), "MB")

## Step 2 — Look at the image

Before measuring anything, look at what you're measuring.

The cell below opens the `.czi` file and pulls out the three channels. It also reads the **pixel size** from the metadata the microscope saved inside the file — that's what lets us report results in real-world units (µm and mm²) instead of pixels.

In [ ]:
# Open a .czi file and return the three channels plus the pixel size in microns
def open_image(path):
    with CziFile(path) as czi:
        # The pixel size is stored in the metadata, in metres. Convert to microns.
        metadata = czi.metadata()
        x_size = float(re.search(r'Distance Id="X">\s*<Value>([^<]+)<', metadata).group(1)) * 1e6

        # Drop any empty dimensions, leaving (channel, y, x)
        data = np.squeeze(czi.asarray())

    # This microscope saves the channels in the order AF488, AF647, DAPI.
    # You'll confirm that visually in the pictures below.
    channels = {"AF488": data[0], "AF647": data[1], "DAPI": data[2]}
    return channels, x_size


channels, pixel_microns = open_image(IMAGE_FILE)

height, width = channels["DAPI"].shape
area_mm2 = (height * width * pixel_microns * pixel_microns) / 1e6

print("Image size:", width, "x", height, "pixels")
print("One pixel is", round(pixel_microns, 3), "microns across")
print("Total imaged area:", round(area_mm2, 3), "mm²")

In [ ]:
plt.figure(figsize=(15, 4))

for position, (name, label_text) in enumerate([("DAPI", "DAPI — every nucleus"),
                                               ("AF488", "AF488 — Olig2 (lineage)"),
                                               ("AF647", "AF647 — Ki67 (dividing)")]):
    image = channels[name]
    plt.subplot(1, 3, position + 1)
    plt.imshow(image, cmap="gray", vmax=np.percentile(image, 99))
    plt.title(label_text)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 💬 Discussion 1 — Reading the three channels

1. The DAPI panel should show many more objects than the other two. Why would you expect that?
2. Compare the AF488 (Olig2) and AF647 (Ki67) panels. Which has fewer bright objects, and what does that tell you biologically?
3. What would you need to see for a cell to count as a *proliferating oligodendrocyte*?



## Step 3 — Detecting cells

A computer doesn't see "cells" — it sees a grid of brightness values. To turn brightness into cell counts we do three things to each channel:

1. **Threshold** — keep only pixels brighter than a chosen cut-off. This is the step that decides what counts as real staining.
2. **Size filter** — discard blobs that are too small (specks of noise) or too large (several nuclei merged together).
3. **Label** — give every remaining separate blob its own number, so we can count them.

> **⚙️ Try it:** the numbers marked ⚙️ below are yours to experiment with. Run this cell and the next one, look at the result, then come back and adjust.

In [ ]:
# Brightness cut-off: a pixel counts as signal only if it is brighter than this
THRESHOLD_488 = 2500     # ⚙️ Try it: Olig2 — higher = stricter, fewer cells
THRESHOLD_647 = 1500     # ⚙️ Try it: Ki67 — higher = stricter, fewer cells

# Size filter, in pixels. Both markers are nuclear, so each object should be
# roughly the size of one nucleus.
MIN_SIZE_488, MAX_SIZE_488 = 300, 5000
MIN_SIZE_647, MAX_SIZE_647 = 100, 3000

# Grow each blob outward by this many pixels to fill small gaps (0 = don't grow)
DILATE_488 = 4           # ⚙️ Try it: lower toward 0 if neighbouring nuclei merge


# Turn one channel into a mask of detected cells, plus a numbered version of it
def find_cells(image, threshold, min_size, max_size, dilate_radius=0):
    mask = image > threshold                       # 1. threshold
    if dilate_radius > 0:
        mask = dilation(mask, disk(dilate_radius))

    labelled = label(mask)                         # 2. number every separate blob

    # 3. keep only the blobs whose size is in range (drops specks and merged clumps)
    good_blobs = [blob.label for blob in regionprops(labelled)
                  if min_size <= blob.area <= max_size]
    keep = np.isin(labelled, good_blobs)

    return keep, label(keep)


mask_488, labels_488 = find_cells(channels["AF488"], THRESHOLD_488,
                                  MIN_SIZE_488, MAX_SIZE_488, DILATE_488)
mask_647, labels_647 = find_cells(channels["AF647"], THRESHOLD_647,
                                  MIN_SIZE_647, MAX_SIZE_647)

print("Olig2+ (AF488) cells detected:", labels_488.max())
print("Ki67+  (AF647) cells detected:", labels_647.max())

Now check the detection by eye. The raw channel is on the left, and what the computer decided was a cell is on the right. **The white shapes should sit on top of the bright nuclei** — not on dim background, and not merged into big blobs.

In [ ]:
plt.figure(figsize=(12, 9))

plt.subplot(2, 2, 1)
plt.imshow(channels["AF488"], cmap="gray", vmax=np.percentile(channels["AF488"], 99))
plt.title("AF488 (Olig2) — raw")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(mask_488, cmap="gray")
plt.title("AF488 (Olig2) — detected cells")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(channels["AF647"], cmap="gray", vmax=np.percentile(channels["AF647"], 99))
plt.title("AF647 (Ki67) — raw")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(mask_647, cmap="gray")
plt.title("AF647 (Ki67) — detected cells")
plt.axis("off")

plt.tight_layout()
plt.show()

### 💬 Discussion 2 — Judging your settings

1. Do the white shapes line up with the bright nuclei? If AF488 is picking up a lot of dim background, would you *raise* or *lower* `THRESHOLD_488`?
2. Raise `THRESHOLD_647` to `3000` and re-run both cells. Did the Ki67 count go up or down? Which way will that push the number of *proliferating* cells you find in Step 4?


## Step 4 — Colocalization: finding the double-positive cells

Now the key step. A pixel counts as **colocalized** only if it belongs to a detected cell in **both** channels at once. We then label those overlap regions and count them.

In [ ]:
# A pixel is colocalized only if it was detected in BOTH channels
coloc_mask = np.logical_and(labels_488 > 0, labels_647 > 0)
coloc_labels = label(coloc_mask)
coloc_count = coloc_labels.max()

print("Olig2+ cells:          ", labels_488.max())
print("Ki67+ cells:           ", labels_647.max())
print("Olig2+Ki67+ cells:     ", coloc_count, "  <- proliferating oligodendrocyte-lineage cells")

In [ ]:
plt.figure(figsize=(13, 4.5))

plt.subplot(1, 3, 1)
plt.imshow(mask_488, cmap="gray")
plt.title("Olig2+ (lineage)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_647, cmap="gray")
plt.title("Ki67+ (dividing)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(coloc_mask, cmap="gray")
plt.title("Overlap = proliferating lineage cells")
plt.axis("off")

plt.tight_layout()
plt.show()

### 💬 Discussion 3 — Where colocalization can mislead

1. Two cells can sit directly on top of each other in a thin tissue section — an Olig2⁺ cell and a *different* Ki67⁺ cell. What would this pipeline call that, and would it be correct?
2. Recall from Step 3 that dilation grows every blob outward by a few pixels. How could that create a false double-positive?



## Step 5 — Turning counts into results

A raw count depends on how much tissue you photographed — a bigger field of view gives more cells regardless of the biology. So we divide by the imaged area to get a **density** in cells per mm².

We also compute the **proliferation fraction**:

$$\text{proliferation fraction} = \frac{\text{Olig2}^{+}\text{Ki67}^{+}\text{ cells}}{\text{all Olig2}^{+}\text{ cells}}$$

This asks what *proportion* of the lineage is dividing, which doesn't depend on how many lineage cells happened to be in the frame.

In [ ]:
results = pd.DataFrame([{
    "Filename": IMAGE_FILE,
    "AF488_Cells": labels_488.max(),
    "AF647_Cells": labels_647.max(),
    "Colocalized_Cells": coloc_count,
    "Image_Area_mm2": round(area_mm2, 4),
    "AF488_cells_per_mm2": round(labels_488.max() / area_mm2, 2),
    "AF647_cells_per_mm2": round(labels_647.max() / area_mm2, 2),
    "Colocalized_cells_per_mm2": round(coloc_count / area_mm2, 2),
    "Proliferation_fraction": round(coloc_count / labels_488.max(), 4),
}])

results.to_csv("colocalization_results.csv", index=False)
print("Saved to colocalization_results.csv")

results.T          # .T flips the table on its side so it's easier to read

### 💬 Discussion 4 — Interpreting your numbers

1. Why do we report cells **per mm²** instead of a raw count?
2. Your proliferation fraction is a small number. Put it into a plain-English sentence a classmate would understand.
3. Imagine a treated group shows a higher density of Olig2⁺Ki67⁺ cells than controls, but a **similar** density of total Olig2⁺ cells. What does that suggest the treatment did?
4. This is **one image**. Give two reasons you shouldn't draw a conclusion about the drug from it.


## Summary

You just ran a complete image-analysis pipeline: **threshold → filter → label → overlap → normalise**, ending with a results table.

Two things to carry forward:

- **Every number here depended on choices you made.** The thresholds, the size filters, the dilation radius — change any of them and the counts change. That isn't a flaw to hide; it's something to set carefully, apply consistently, and report.
- **A count becomes a result only after normalising.** Cells per mm² and the proliferation fraction are what make two images comparable.

In the next three notebooks you'll take a table exactly like this one — pooled across the whole class's images — and learn how to describe it, summarise it, and compare CON against DRUG.

---
## Nice work — you're done!

Before you export: check that you've answered every **💬 Discussion** cell, then click **Runtime → Run all**, then **File → Print → Save as PDF** and upload the PDF to Canvas.